# Day 7: Incremental Processing & CDC Upsert Engine
This notebook simulates idempotent UPSERT/DELETE logic on a simulated stateful table.

In [ ]:
!pip install pyspark pyiceberg duckdb splink

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day07_CDC_Merge") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.silver")

# 1. Initialize target table (Silver)
spark.sql("""
CREATE TABLE IF NOT EXISTS local.silver.patients (
    payload_id STRING,
    name STRING,
    status STRING,
    updated_at TIMESTAMP
)
USING iceberg
""")

# Insert initial state
spark.sql("""
INSERT INTO local.silver.patients VALUES
('1', 'Alice', 'Active', timestamp '2023-01-01 10:00:00'),
('2', 'Bob', 'Active', timestamp '2023-01-01 10:00:00')
""")
print("INITIAL STATE:")
spark.sql("SELECT * FROM local.silver.patients ORDER BY payload_id").show()

# 2. New CDC Batch (Updates Bob, Deletes Alice, Inserts Charlie, stale update for Bob ignored)
spark.sql("""
CREATE OR REPLACE TEMP VIEW cdc_batch AS SELECT * FROM VALUES
('1', 'Alice', 'Inactive', timestamp '2023-01-02 10:00:00', 'd'),  -- Delete Alice
('2', 'Bob', 'Inactive', timestamp '2023-01-02 10:00:00', 'u'),    -- Valid update for Bob
('2', 'Bob', 'Active', timestamp '2022-12-31 10:00:00', 'u'),      -- Stale out-of-order update for Bob (should be ignored)
('3', 'Charlie', 'Active', timestamp '2023-01-02 10:00:00', 'c')   -- Insert Charlie
AS t(payload_id, name, status, updated_at, _cdc_op)
""")

# 3. Execute MERGE INTO
spark.sql("""
MERGE INTO local.silver.patients t
USING cdc_batch s
ON t.payload_id = s.payload_id
WHEN MATCHED AND s._cdc_op = 'd' AND s.updated_at >= t.updated_at THEN 
    DELETE
WHEN MATCHED AND s._cdc_op IN ('c', 'u') AND s.updated_at >= t.updated_at THEN 
    UPDATE SET *
WHEN NOT MATCHED AND s._cdc_op IN ('c', 'u') THEN 
    INSERT (payload_id, name, status, updated_at) VALUES (s.payload_id, s.name, s.status, s.updated_at)
""")

print("STATE AFTER CDC MERGE:")
spark.sql("SELECT * FROM local.silver.patients ORDER BY payload_id").show()
